# 27: Hugging Face Transformers - Using Pre-Trained Models

## Standing on the Shoulders of Giants

Training Transformers from scratch requires:
- Massive datasets (billions of words)
- Expensive GPUs (months of training)
- Millions of dollars

**Solution**: Use **pre-trained models** from Hugging Face!
- Thousands of models ready to use
- Fine-tune for your specific task
- Transfer learning at its best

### The Web Dev Analogy

Hugging Face is like **npm/PyPI for ML**:
- **Model Hub**: Package registry (thousands of models)
- **Transformers library**: Easy API (like Express.js)
- **Datasets library**: Data loaders (like axios)
- **Community**: Contributions and sharing

Don't build from scratch—use battle-tested models!

In [ ]:
# Install transformers if needed
# !pip install transformers

import torch
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    pipeline
)
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to use Hugging Face! 🤗")

## 1. The Transformers Library

**Key concepts:**
1. **Model**: Pre-trained Transformer
2. **Tokenizer**: Convert text ↔ tokens
3. **Pipeline**: High-level API for common tasks
4. **Auto classes**: Automatically load correct model/tokenizer

In [ ]:
print("Hugging Face Ecosystem:")
print("=" * 70)

print("\n📦 transformers: Core library")
print("   - Load pre-trained models")
print("   - Tokenizers for text processing")
print("   - Training utilities")

print("\n🗂️  datasets: Easy data loading")
print("   - 10,000+ datasets")
print("   - Efficient loading and processing")

print("\n🚀 accelerate: Distributed training")
print("   - Multi-GPU training")
print("   - Mixed precision")

print("\n🏠 Model Hub: hub.huggingface.co")
print("   - 100,000+ models")
print("   - Version control")
print("   - Model cards (documentation)")

## 2. Using Pipelines (Easiest Way)

In [ ]:
# Sentiment analysis pipeline
classifier = pipeline("sentiment-analysis")

texts = [
    "I love this movie! It's amazing!",
    "This is terrible, worst film ever.",
    "It was okay, not great but not bad."
]

results = classifier(texts)

print("Sentiment Analysis:")
print("=" * 70)
for text, result in zip(texts, results):
    print(f"\nText: '{text}'")
    print(f"  → {result['label']}: {result['score']:.4f}")

print("\n✅ Just 2 lines of code for sentiment analysis!")

In [ ]:
# Text generation pipeline
generator = pipeline("text-generation", model="gpt2")

prompt = "Artificial intelligence will"
outputs = generator(
    prompt,
    max_length=50,
    num_return_sequences=3,
    temperature=0.8
)

print("Text Generation:")
print("=" * 70)
print(f"Prompt: '{prompt}'\n")

for i, output in enumerate(outputs, 1):
    print(f"{i}. {output['generated_text']}")
    print()

print("💡 GPT-2 generating text automatically!")

`★ Insight ─────────────────────────────────────`

**Available pipelines:**
- `sentiment-analysis`: Positive/negative classification
- `text-generation`: Auto-complete text
- `fill-mask`: Fill in [MASK] tokens (like BERT)
- `question-answering`: Answer questions from context
- `summarization`: Summarize long text
- `translation`: Translate between languages
- `zero-shot-classification`: Classify without training

Pipelines handle tokenization, inference, and decoding automatically!

`─────────────────────────────────────────────────`

## 3. Tokenizers

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Hello, how are you doing today?"

# Tokenize
tokens = tokenizer.tokenize(text)
print(f"Text: {text}")
print(f"\nTokens: {tokens}")

# Convert to IDs
input_ids = tokenizer.encode(text)
print(f"\nToken IDs: {input_ids}")

# Full encoding (with attention mask, etc.)
encoding = tokenizer(
    text,
    return_tensors="pt",  # Return PyTorch tensors
    padding=True,
    truncation=True
)

print(f"\nFull encoding:")
for key, value in encoding.items():
    print(f"  {key}: {value.shape}")

# Decode back
decoded = tokenizer.decode(input_ids)
print(f"\nDecoded: {decoded}")

In [ ]:
# Special tokens
print("Special Tokens:")
print("=" * 70)
print(f"CLS token (start of sequence): {tokenizer.cls_token} → {tokenizer.cls_token_id}")
print(f"SEP token (separator): {tokenizer.sep_token} → {tokenizer.sep_token_id}")
print(f"PAD token (padding): {tokenizer.pad_token} → {tokenizer.pad_token_id}")
print(f"UNK token (unknown): {tokenizer.unk_token} → {tokenizer.unk_token_id}")
print(f"MASK token (for MLM): {tokenizer.mask_token} → {tokenizer.mask_token_id}")

print(f"\nVocabulary size: {tokenizer.vocab_size:,}")

## 4. Using Pre-trained Models

In [ ]:
# Load BERT model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

print(f"Loaded model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Prepare input
text = "The Transformer architecture is powerful."
inputs = tokenizer(text, return_tensors="pt")

print(f"\nInput text: '{text}'")
print(f"Input IDs shape: {inputs['input_ids'].shape}")

# Forward pass
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Get embeddings
last_hidden_state = outputs.last_hidden_state
print(f"\nOutput shape: {last_hidden_state.shape}")
print(f"  (batch_size, sequence_length, hidden_size)")

# Use [CLS] token for sentence representation
cls_embedding = last_hidden_state[:, 0, :]
print(f"\n[CLS] embedding shape: {cls_embedding.shape}")
print(f"  → 768-dimensional sentence representation")

## 5. Fine-tuning for Classification

In [ ]:
# Load model for classification
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2  # Binary classification
)

print(f"Model for classification:")
print(f"  Base: BERT")
print(f"  Task: Sequence classification")
print(f"  Num labels: 2")
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

# The model adds a classification head on top of BERT
print(f"\nClassification head added:")
print(f"  [CLS] embedding (768) → Linear → 2 logits")

In [ ]:
# Simple training loop structure
print("Fine-tuning Structure:")
print("=" * 70)
print("""
# 1. Prepare data
train_dataset = ...
val_dataset = ...

# 2. Setup
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3

# 3. Training loop
for epoch in range(num_epochs):
    model.train()
    for batch in train_dataloader:
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    # Validation
    model.eval()
    ...

# 4. Save
model.save_pretrained('./my-model')
tokenizer.save_pretrained('./my-model')
""")

print("\n💡 Use Trainer API for even easier training!")

`★ Insight ─────────────────────────────────────`

**Transfer learning workflow:**
1. **Pre-training**: Model learns language on massive data (done for you!)
2. **Fine-tuning**: Adapt to your specific task (what you do)
   - Much less data needed
   - Much faster training
   - Better results than training from scratch

This is why modern NLP is so accessible!

`─────────────────────────────────────────────────`

## 6. Popular Models

In [ ]:
print("Popular Pre-trained Models:")
print("=" * 70)

models = {
    "BERT": {
        "type": "Encoder-only",
        "use_case": "Classification, NER, QA",
        "models": ["bert-base-uncased", "bert-large-uncased"],
        "size": "110M - 340M parameters"
    },
    "RoBERTa": {
        "type": "Encoder-only (improved BERT)",
        "use_case": "Classification, better than BERT",
        "models": ["roberta-base", "roberta-large"],
        "size": "125M - 355M parameters"
    },
    "GPT-2": {
        "type": "Decoder-only",
        "use_case": "Text generation",
        "models": ["gpt2", "gpt2-medium", "gpt2-large"],
        "size": "117M - 1.5B parameters"
    },
    "T5": {
        "type": "Encoder-Decoder",
        "use_case": "Any task (unified framework)",
        "models": ["t5-small", "t5-base", "t5-large"],
        "size": "60M - 11B parameters"
    },
    "DistilBERT": {
        "type": "Encoder-only (distilled BERT)",
        "use_case": "Fast inference, 97% of BERT",
        "models": ["distilbert-base-uncased"],
        "size": "66M parameters"
    }
}

for name, info in models.items():
    print(f"\n{name}:")
    for key, value in info.items():
        if key == "models":
            print(f"  {key}: {', '.join(value)}")
        else:
            print(f"  {key}: {value}")

## 7. Model Hub

In [ ]:
print("Hugging Face Model Hub:")
print("=" * 70)
print("\n🌐 Website: huggingface.co/models")

print("\n📝 Features:")
print("  - 100,000+ models")
print("  - Model cards (documentation)")
print("  - Try models in browser (Inference API)")
print("  - Version control with Git")
print("  - Download counts and likes")

print("\n🔍 Search by:")
print("  - Task (classification, generation, etc.)")
print("  - Language (English, Chinese, etc.)")
print("  - Framework (PyTorch, TensorFlow, JAX)")
print("  - Dataset (trained on what data)")
print("  - License")

print("\n💾 Usage:")
print("  from_pretrained('username/model-name')")
print("  → Automatically downloads and caches")

print("\n📤 Upload your own:")
print("  model.push_to_hub('my-awesome-model')")
print("  → Share with the community!")

## 8. Best Practices

In [ ]:
print("Best Practices for Using Transformers:")
print("=" * 70)

print("\n1. Model Selection:")
print("   ✓ Start with a small model (distilbert, bert-base)")
print("   ✓ Use task-specific models when available")
print("   ✓ Check model card for capabilities/limitations")
print("   ✓ Consider model size vs accuracy tradeoff")

print("\n2. Data Preparation:")
print("   ✓ Use the model's tokenizer (not generic)")
print("   ✓ Respect max sequence length")
print("   ✓ Use padding and truncation appropriately")
print("   ✓ Batching for efficiency")

print("\n3. Fine-tuning:")
print("   ✓ Low learning rate (1e-5 to 5e-5)")
print("   ✓ Few epochs (2-4 usually enough)")
print("   ✓ Use validation set to prevent overfitting")
print("   ✓ Save checkpoints")

print("\n4. Inference:")
print("   ✓ Use model.eval() mode")
print("   ✓ Disable gradients (torch.no_grad())")
print("   ✓ Batch predictions when possible")
print("   ✓ Consider quantization for production")

print("\n5. Common Pitfalls:")
print("   ✗ Using wrong tokenizer for model")
print("   ✗ Forgetting special tokens ([CLS], [SEP])")
print("   ✗ High learning rate (destroys pre-training)")
print("   ✗ Too many epochs (overfitting)")
print("   ✗ Not checking model license")

## 9. Quick Start Template

In [ ]:
print("Quick Start Template:")
print("=" * 70)
print("""
# ---- For Classification ----

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Load
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# 2. Prepare data
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

# 3. Train
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

# 4. Use
inputs = tokenizer("Your text here", return_tensors='pt')
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=-1)
""")

print("\n💡 This pattern works for most tasks!")

## 📝 Check Your Understanding

1. What is the Hugging Face Model Hub?
2. What does a tokenizer do?
3. What's the difference between AutoModel and AutoModelForSequenceClassification?
4. Why is transfer learning powerful?
5. When would you use a pipeline vs loading a model directly?

## 🎯 Summary

**Hugging Face Transformers**:
- **Library**: Easy interface to pre-trained models
- **Model Hub**: 100,000+ ready-to-use models
- **Transfer Learning**: Pre-train → Fine-tune paradigm

**Key components**:
1. **Tokenizer**: Text → token IDs
2. **Model**: Pre-trained Transformer
3. **Pipeline**: High-level API for common tasks
4. **Auto classes**: Automatically load correct components

**Usage patterns**:
```python
# Simple (pipeline)
pipe = pipeline('sentiment-analysis')
pipe("I love this!")

# Advanced (direct model)
tokenizer = AutoTokenizer.from_pretrained('bert-base')
model = AutoModel.from_pretrained('bert-base')
inputs = tokenizer(text, return_tensors='pt')
outputs = model(**inputs)
```

**Popular models**:
- **BERT**: Classification, NER
- **GPT-2**: Text generation
- **T5**: Any task (encoder-decoder)
- **DistilBERT**: Fast inference

**Why it matters**:
- No need to train from scratch
- State-of-the-art results with little data
- Easy to experiment and deploy
- Active community and support

**Next steps**:
1. Explore Model Hub
2. Try different pipelines
3. Fine-tune for your task
4. Share your models!

## 🎉 Congratulations!

You've completed the ML curriculum from zero to Transformers!
- ✅ NumPy and data fundamentals
- ✅ Classification and regression
- ✅ Neural networks from scratch
- ✅ PyTorch and training loops
- ✅ Word embeddings
- ✅ RNNs and LSTMs
- ✅ Attention mechanism
- ✅ Transformers
- ✅ Pre-trained models

You now have the foundation to build modern NLP applications! 🚀